# Reversed-Digit Addition: Length-Generalization Validation

**Goal (harness validation, not a research result).** Reproduce the known effect that a small
from-scratch decoder-only transformer on reversed-digit addition learns the task in-distribution
with any positional encoding, but length-generalizes very differently out-of-distribution:
learned **absolute** PE collapses toward 0 on longer operands, while **NoPE** and **RoPE** degrade
gracefully.

## Success criteria

1. In-distribution exact-match >= ~0.95 for all three PE variants.
2. OOD: absolute PE collapses toward 0 as length grows; NoPE (ideally RoPE) stays materially higher on the first OOD lengths.
3. The accuracy-vs-length curve visibly separates the variants past length 5.

Run order: train the three variants first (`bash scripts/train_all_variants.sh`), then Restart & Run All here.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Make `src` importable whether the kernel starts in the repo root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.datasets.addition_dataset import AdditionDataset
from src.datasets.collate import collate_fn
from src.datasets.tokenizer import Tokenizer
from src.evaluation.length_generalization import exact_match_at_length, sweep_lengths
from src.model.transformer import DecoderTransformer
from src.utils.init_utils import set_random_seed

SEED = 12345
set_random_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = Tokenizer()
tokenizer.print_vocab()
print("repo root:", ROOT)
print("device:", device)

## 1. Format verification (before trusting anything)

Three decoded examples in the reversed-digit format, plus the loss mask for one
example showing that **only the answer tokens** (`rev(s)` and `<eos>`) contribute
to the loss.

In [ ]:
demo = AdditionDataset(min_len=1, max_len=5, dataset_length=8, split="eval", seed=SEED)
items = [demo[i] for i in range(3)]

print("Decoded examples  (rev(a) + rev(b) = rev(s) <eos>):")
for it in items:
    ids = it["input_ids"].tolist()
    print("   ", " ".join(tokenizer.decode(ids, stop_at_eos=False, strip_special=False)))

print()
print("Loss-mask alignment for example 0 (mask=1 -> contributes to the loss):")
batch = collate_fn(items)
ids0 = batch["input_ids"][0].tolist()
mask0 = batch["loss_mask"][0].tolist()
for sym, m in zip(tokenizer.decode(ids0, stop_at_eos=False, strip_special=False), mask0):
    print(f"    {sym:>5}   mask={m}")

## 2. Load the trained checkpoints

Expects `saved/addition_{absolute,nope,rope}/model_best.pth` from
`bash scripts/train_all_variants.sh`. Missing checkpoints are skipped with a note.

In [ ]:
VARIANTS = ["absolute", "nope", "rope"]


def load_model(variant):
    ckpt_path = ROOT / "saved" / f"addition_{variant}" / "model_best.pth"
    if not ckpt_path.exists():
        print(f"[missing] {ckpt_path} -- train this variant first (see README).")
        return None
    ckpt = torch.load(ckpt_path, map_location=device)
    model = DecoderTransformer(
        vocab_size=tokenizer.vocab_size, d_model=128, n_layers=4, n_heads=2,
        head_dim=64, d_ff=512, max_len=64, pe_variant=variant, pad_id=tokenizer.pad_id,
    )
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
    model.load_state_dict(state)
    return model.to(device).eval()


models = {v: load_model(v) for v in VARIANTS}
available = [v for v, m in models.items() if m is not None]
print("loaded variants:", available)

## 3. Per-length exact-match sweep (autoregressive greedy)

Lengths 1-5 are in-distribution; 6-15 are out-of-distribution. Each length uses a
fixed, deterministic eval set, so all variants are compared on **identical** examples.
This is the authoritative metric (free-running greedy decoding), distinct from the
teacher-forced monitor used during training.

In [ ]:
IN_DIST = list(range(1, 6))   # in-distribution lengths
OOD = list(range(6, 16))      # out-of-distribution lengths
LENGTHS = IN_DIST + OOD
N_EVAL = 200                  # examples per length (low hundreds)

data = {}
for v in available:
    accs = sweep_lengths(
        models[v], LENGTHS, n=N_EVAL, seed=SEED, device=device, tokenizer=tokenizer
    )
    data[v] = [accs[L] for L in LENGTHS]

results = pd.DataFrame(data, index=LENGTHS)
results.index.name = "operand_length"
results.round(3)

## 4. The money plot

Exact-match accuracy vs operand length, one line per PE variant, with the
train/OOD boundary marked. Saved to `figures/money_plot.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for v in results.columns:
    ax.plot(results.index, results[v], marker="o", label=v)
ax.axvline(5.5, ls="--", color="gray", alpha=0.8, label="train / OOD boundary")
ax.set_xlabel("operand length (digits)")
ax.set_ylabel("exact-match accuracy")
ax.set_title("Length generalization on reversed-digit addition")
ax.set_ylim(-0.02, 1.02)
ax.set_xticks(LENGTHS)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

out_path = ROOT / "figures" / "money_plot.png"
fig.savefig(out_path, dpi=150)
print("saved", out_path)
plt.show()

## 5. Decoded failure cases (absolute PE at an OOD length)

Makes the failure mode concrete: the model typically breaks down once positions
exceed those seen in training.

In [ ]:
if models.get("absolute") is not None:
    acc10, examples = exact_match_at_length(
        models["absolute"], length=10, n=200, seed=SEED, device=device,
        tokenizer=tokenizer, return_examples=10,
    )
    print(f"absolute PE exact-match @ length 10: {acc10:.3f}\n")
    for ex in examples:
        flag = "OK" if ex["correct"] else "XX"
        print(f"  [{flag}] {ex['a']} + {ex['b']}  pred={ex['pred']!r}  target={ex['target']!r}")
else:
    print("absolute checkpoint not available -- train it first.")

## 6. Validation verdict

Mechanically checks the success criteria from the build plan (§0) and prints
**PASS / FAIL** with the numbers.

In [ ]:
def check_criteria(results):
    cols = list(results.columns)
    indist = results.loc[IN_DIST]

    print("Criterion 1 -- in-distribution exact match (mean over lengths 1-5):")
    c1 = True
    for v in cols:
        m = indist[v].mean()
        ok = m >= 0.95
        c1 = c1 and ok
        print(f"    {v:>9}: {m:.3f}  {'OK' if ok else 'LOW'}")
    if len(cols) < 3:
        print("    (note: fewer than 3 variants loaded)")

    print()
    print("Criterion 2 -- OOD gap NoPE minus absolute (mean over lengths 6-8):")
    c2 = None
    if "nope" in cols and "absolute" in cols:
        gap = (results.loc[[6, 7, 8], "nope"] - results.loc[[6, 7, 8], "absolute"]).mean()
        c2 = gap > 0.2
        print(f"    gap = {gap:.3f}  {'OK' if c2 else 'SMALL'}")
    else:
        print("    need both nope and absolute variants loaded")

    print()
    print("Criterion 3 -- absolute PE collapses by length 10:")
    c3 = None
    if "absolute" in cols:
        a10 = results.loc[10, "absolute"]
        c3 = a10 < 0.2
        print(f"    absolute EM @10 = {a10:.3f}  {'OK' if c3 else 'HIGH'}")
    else:
        print("    absolute variant not loaded")

    passed = bool(c1) and (c2 is True) and (c3 is True)
    print()
    print("=" * 48)
    print("VERDICT:", "PASS -- harness validated" if passed else "FAIL / INCOMPLETE")
    print("=" * 48)
    if not passed:
        print("If FAIL, debug in priority order: loss masking -> data format ->")
        print("training budget -> model bug.")
    print("Single seed here; the full study should repeat across >= 3 seeds.")
    return passed


_ = check_criteria(results)